In [1]:
import json
import numpy as np
import plotly.graph_objects as go
from utils import *

In [2]:
def save_calibration_panoptic(output_file, extrinsic_matrices, camera_matrices, distortion_coefficients):
    customized_sequence = []
    num_cameras = extrinsic_matrices.shape[0]

    for i in range(num_cameras):
        # 1. Extract Rotation (3x3) and Translation (3x1)
        # Note: Panoptic format usually expects R and T from the extrinsic matrix
        R = extrinsic_matrices[i, :3, :3].tolist()
        T = extrinsic_matrices[i, :3, 3:4].tolist() # Keeps it as [[x], [y], [z]]

        # 2. Extract Intrinsics
        K = camera_matrices[i]
        fx = float(K[0, 0])
        fy = float(K[1, 1])
        cx = float(K[0, 2])
        cy = float(K[1, 2])

        # 3. Extract Distortion (k1, k2, k3) and (p1, p2)
        # OpenCV format: [k1, k2, p1, p2, k3]
        dist = distortion_coefficients[i]
        k = [[float(dist[0])], [float(dist[1])], [float(dist[4])]]
        p = [[float(dist[2])], [float(dist[3])]]

        # Create camera dictionary
        cam_data = {
            "R": R,
            "T": T,
            "fx": fx,
            "fy": fy,
            "cx": cx,
            "cy": cy,
            "k": k,
            "p": p
        }
        customized_sequence.append(cam_data)

    output_data = {
        "customized_sequence": customized_sequence
    }

    with open(output_file, 'w') as f:
        json.dump(output_data, f, indent=4)
    
    print(f"Successfully saved calibration to {output_file}")

def visualize_panoptic_setup(cameras):
    fig = go.Figure()
    all_points = [] 

    # In Panoptic dataset, T is usually the world position (mm)
    centers = np.array([np.array(cam['T']).flatten() for cam in cameras])
    if len(centers) > 1:
        avg_dist = np.mean(np.linalg.norm(centers[:, None] - centers, axis=2))
        frustum_depth = avg_dist * 0.1
    else:
        frustum_depth = 300.0

    for i, cam in enumerate(cameras):
        R = np.array(cam['R'])
        T = np.array(cam['T']).flatten()
        C = T  
        
        fx, fy = cam['fx'], cam['fy']
        cx, cy = cam['cx'], cam['cy']
        w, h = cx * 2, cy * 2 

        corners_cam = np.array([
            [(0 - cx) * frustum_depth / fx, (0 - cy) * frustum_depth / fy, frustum_depth],
            [(w - cx) * frustum_depth / fx, (0 - cy) * frustum_depth / fy, frustum_depth],
            [(w - cx) * frustum_depth / fx, (h - cy) * frustum_depth / fy, frustum_depth],
            [(0 - cx) * frustum_depth / fx, (h - cy) * frustum_depth / fy, frustum_depth]
        ])
        
        corners_world = (R.T @ corners_cam.T).T + T
        all_points.extend(corners_world)
        all_points.append(C)

        for corner in corners_world:
            fig.add_trace(go.Scatter3d(
                x=[C[0], corner[0]], y=[C[1], corner[1]], z=[C[2], corner[2]],
                mode='lines', line=dict(color='blue', width=2), showlegend=False
            ))
        
        rect = np.vstack([corners_world, corners_world[0]])
        fig.add_trace(go.Scatter3d(
            x=rect[:, 0], y=rect[:, 1], z=rect[:, 2],
            mode='lines', line=dict(color='blue', width=2), showlegend=False
        ))

        fig.add_trace(go.Mesh3d(
            x=corners_world[:, 0], y=corners_world[:, 1], z=corners_world[:, 2],
            i=[0, 0], j=[1, 2], k=[2, 3], 
            opacity=0.15, color='cyan', name=f'Cam {i} FOV'
        ))

        fig.add_trace(go.Scatter3d(
            x=[C[0]], y=[C[1]], z=[C[2]],
            mode='markers+text', marker=dict(size=4, color='red'),
            text=[f"Cam {i}"], textposition="top center", name=f"Camera {i}"
        ))


    # Create the cone
    cam_positions = np.array([np.array(cam['T']).flatten() for cam in cameras])
    center_of_cameras = np.mean(cam_positions, axis=0)
    print(f"Center of Cameras: {center_of_cameras}")
    cone_height = frustum_depth * 2.0  # Make it prominent
    cone_base_radius = frustum_depth * 0.5
    theta = np.linspace(0, 2*np.pi, 20)
    bx = center_of_cameras[0] + cone_base_radius * np.cos(theta)
    by = center_of_cameras[1] + cone_base_radius * np.sin(theta)
    bz = np.full_like(theta, center_of_cameras[2])
    tip = [center_of_cameras[0], center_of_cameras[1], center_of_cameras[2] + cone_height]
    x_cone = np.append(bx, tip[0])
    y_cone = np.append(by, tip[1])
    z_cone = np.append(bz, tip[2])
    i_indices = []
    j_indices = []
    k_indices = []
    for n in range(len(theta) - 1):
        i_indices.append(n)
        j_indices.append(n + 1)
        k_indices.append(20) # Connect to tip
        
    fig.add_trace(go.Mesh3d(
        x=x_cone, y=y_cone, z=z_cone,
        i=i_indices, j=j_indices, k=k_indices,
        color='green', opacity=0.4, name='Z-Axis Pointer'
    ))

    all_points = np.array(all_points)
    center_scene = (all_points.min(axis=0) + all_points.max(axis=0)) / 2
    max_range = np.ptp(all_points, axis=0).max() / 2

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[center_scene[0]-max_range, center_scene[0]+max_range], title='X (mm)'),
            yaxis=dict(range=[center_scene[1]-max_range, center_scene[1]+max_range], title='Y (mm)'),
            zaxis=dict(range=[center_scene[2]-max_range, center_scene[2]+max_range], title='Z (mm)'),
            aspectmode='cube'
        ),
        title="Panoptic Studio Setup)",
        margin=dict(l=0, r=0, b=0, t=50)
    )

    fig.show()

In [ ]:
cameras_raw = read_calibration_raw('calibration_raw.json')
extrinsic_matrices = extract_extrinsics(cameras_raw)    # Shape (n, 4, 4) <class 'numpy.ndarray'>
extrinsic_matrices = invert_extrinsics(extrinsic_matrices)
camera_matrices = extract_camera_matrices(cameras_raw)  # Shape (n, 3, 3) <class 'numpy.ndarray'>
distortion_coefficients = extract_distortion_coefficients(cameras_raw) # Shape (n, 5) <class 'numpy.ndarray'>

save_calibration_panoptic('calibration.json', extrinsic_matrices, camera_matrices, distortion_coefficients)

# --

cameras_panoptic = read_calibration_panoptic('calibration.json')
visualize_panoptic_setup(cameras_panoptic)

Successfully saved calibration to calibration.json
Center of Cameras: [0.20138191 0.45089457 2.73793881]
